# Lezione 04 — Pratica: Hadoop, Hive, Spark e data mining

Nella lezione di teoria abbiamo visto quale problema risolvono Hadoop, HDFS, Hive e Spark, il processo di data mining in 6 passi, e abbiamo programmato un MapReduce giocattolo per contare parole. Ora tocca a te completare il codice.

## Esercizio 1: MapReduce con filtro delle stopword e distribuzione reale sui nodi

Migliorerai il MapReduce della lezione: ci sono parole molto frequenti ("il", "la", "di"...) che non apportano informazione — si chiamano **stopword** — e per ora il conteggio le include, sporcando il risultato. Inoltre, la funzione che distribuisce i documenti tra "nodi" non è ancora implementata.

**Il tuo compito**:
1. Completa `map_palabras` perché **non** includa nel risultato le parole presenti in `STOPWORDS`.
2. Completa `repartir_en_nodos` perché distribuisca la lista `documentos` in `n_nodos` sottoliste usando slicing (come visto in lezione).

Suggerimento per il passo 2: `documentos[i:i + tamano_trozo]` scorrendo `i` a salti di `tamano_trozo`, proprio come fatto nella demo di lezione — ma ora devi scriverlo tu dentro una funzione riutilizzabile.

In [ ]:
documentos = [
    "il gatto dorme sul tappeto tutto il giorno",
    "i cani abbaiano quando passa il postino per strada",
    "un gatto e un cane possono essere buoni amici se vengono presentati bene",
    "il tappeto del salotto è il posto preferito del gatto",
    "il postino porta pacchi tutti i giorni tranne la domenica",
    "i cani e i gatti dormono molto piu degli umani",
]

STOPWORDS = {"il", "la", "i", "gli", "le", "un", "una", "di", "in",
             "del", "per", "sul", "e", "che", "se", "tutto", "tutti", "e'", "degli"}


def map_palabras(texto):
    """Fase MAP: restituisce coppie (parola, 1), ESCLUDENDO le
    parole presenti in STOPWORDS."""
    palabras = texto.split()
    pares = []
    for palabra in palabras:
        # TODO: aggiungi la coppia (palabra, 1) solo se `palabra` NON è in STOPWORDS
        pares.append((palabra, 1))
    return pares


def reduce_conteos(pares):
    """Fase REDUCE: è già completa, non serve modificarla."""
    conteos = {}
    for palabra, valor in pares:
        conteos[palabra] = conteos.get(palabra, 0) + valor
    return conteos


def repartir_en_nodos(lista_documentos, n_nodos):
    """Distribuisce lista_documentos in n_nodos sottoliste quasi uguali,
    simulando come HDFS distribuirebbe i dati tra i nodi."""
    # TODO: calcola tamano_trozo e restituisci la lista di sottoliste
    # usando slicing, come fatto nella demo di lezione.
    return [lista_documentos]  # placeholder: per ora, "1 solo nodo" con tutto


# Con il placeholder sopra, questo già funziona (anche se il risultato
# non è ancora corretto perché i TODO non sono risolti):
nodos = repartir_en_nodos(documentos, 3)
print(f"Numero di nodi simulati: {len(nodos)}")

pares_intermedios = []
for datos_nodo in nodos:
    for doc in datos_nodo:
        pares_intermedios.extend(map_palabras(doc))

resultado = reduce_conteos(pares_intermedios)
top_5 = sorted(resultado.items(), key=lambda par: par[1], reverse=True)[:5]
print("Top 5 (prima di risolvere i TODO):", top_5)

<details>
<summary>💡 Vedi soluzione</summary>

```python
documentos = [
    "il gatto dorme sul tappeto tutto il giorno",
    "i cani abbaiano quando passa il postino per strada",
    "un gatto e un cane possono essere buoni amici se vengono presentati bene",
    "il tappeto del salotto è il posto preferito del gatto",
    "il postino porta pacchi tutti i giorni tranne la domenica",
    "i cani e i gatti dormono molto piu degli umani",
]

STOPWORDS = {"il", "la", "i", "gli", "le", "un", "una", "di", "in",
             "del", "per", "sul", "e", "che", "se", "tutto", "tutti", "e'", "degli"}


def map_palabras(texto):
    palabras = texto.split()
    pares = []
    for palabra in palabras:
        if palabra not in STOPWORDS:  # <- filtro delle stopword
            pares.append((palabra, 1))
    return pares


def reduce_conteos(pares):
    conteos = {}
    for palabra, valor in pares:
        conteos[palabra] = conteos.get(palabra, 0) + valor
    return conteos


def repartir_en_nodos(lista_documentos, n_nodos):
    tamano_trozo = len(lista_documentos) // n_nodos + 1
    return [
        lista_documentos[i:i + tamano_trozo]
        for i in range(0, len(lista_documentos), tamano_trozo)
    ]


nodos = repartir_en_nodos(documentos, 3)
print(f"Numero di nodi simulati: {len(nodos)}")

pares_intermedios = []
for datos_nodo in nodos:
    for doc in datos_nodo:
        pares_intermedios.extend(map_palabras(doc))

resultado = reduce_conteos(pares_intermedios)
top_5 = sorted(resultado.items(), key=lambda par: par[1], reverse=True)[:5]
print("Top 5 (ora senza stopword):", top_5)
```
</details>

## Esercizio 2: individuare i 6 passi del data mining

Un'azienda di retail vuole ridurre l'abbandono dei clienti (churn). Di seguito trovi una lista di attività svolte dal loro team dati, **in ordine**, ma senza l'etichetta del passo del processo di data mining a cui corrispondono.

**Il tuo compito**: completa la funzione `identificar_paso`, che riceve la descrizione di un'attività (in minuscolo) e restituisce il numero del passo (da 1 a 6) a cui corrisponde, usando parole chiave. Ti lasciamo già risolti i passi 1 e 6 come esempio — completa il resto seguendo la stessa logica.

In [ ]:
actividades = [
    "definiamo che vogliamo predire quali clienti cancelleranno il loro abbonamento",
    "raccogliamo dati di fatturazione, assistenza tecnica e uso dell app",
    "controlliamo i dati e troviamo colonne con valori impossibili e duplicati",
    "convertiamo le date in un formato unico e normalizziamo le valute",
    "addestriamo un albero decisionale per predire la cancellazione",
    "misuriamo l accuratezza del modello e presentiamo i risultati al team dirigenziale",
]

PASOS = {
    1: "Fissare l'obiettivo",
    2: "Selezionare le fonti di dati",
    3: "Pre-elaborare",
    4: "Trasformare",
    5: "Estrarre (mining)",
    6: "Valutare",
}


def identificar_paso(descripcion):
    """Restituisce il numero del passo (1-6) in base alle parole chiave
    presenti nella descrizione dell'attività."""
    if "definiamo" in descripcion or "vogliamo predire" in descripcion:
        return 1
    # TODO: passo 2 — parole chiave come "raccogliamo" o "dati di"
    # TODO: passo 3 — parole chiave come "controlliamo", "duplicati", "valori impossibili"
    # TODO: passo 4 — parole chiave come "convertiamo", "normalizziamo", "formato"
    # TODO: passo 5 — parole chiave come "addestriamo", "modello", "albero"
    if "misuriamo" in descripcion or "presentiamo" in descripcion:
        return 6
    return 0  # ancora non classificato


for actividad in actividades:
    paso = identificar_paso(actividad)
    nombre_paso = PASOS.get(paso, "NON CLASSIFICATO")
    print(f"[{nombre_paso}] {actividad}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
def identificar_paso(descripcion):
    if "definiamo" in descripcion or "vogliamo predire" in descripcion:
        return 1
    elif "raccogliamo" in descripcion or "dati di" in descripcion:
        return 2
    elif "controlliamo" in descripcion or "duplicati" in descripcion or "valori impossibili" in descripcion:
        return 3
    elif "convertiamo" in descripcion or "normalizziamo" in descripcion or "formato" in descripcion:
        return 4
    elif "misuriamo" in descripcion or "presentiamo" in descripcion:
        return 6
    elif "addestriamo" in descripcion or "albero" in descripcion:
        return 5
    return 0


for actividad in actividades:
    paso = identificar_paso(actividad)
    nombre_paso = PASOS.get(paso, "NON CLASSIFICATO")
    print(f"[{nombre_paso}] {actividad}")
```
</details>

## Per tornare

Vuoi ripassare la teoria prima di continuare? Torna al notebook di lezione: [clase04_hadoop_hive_spark_IT.ipynb](../notebooks_clase/clase04_hadoop_hive_spark_IT.ipynb)